# Data Provenance Metadata in DTC-IS

This notebook is a demonstrator from the **Digital Twin Component for Ice Sheets (DTC-IS)**, an ESA-funded, EO-driven digital twin project.

DTC-IS integrates Earth Observation, in-situ data, reanalysis products, and advanced data-driven models to provide a dynamic and science-based representation of the Greenland and Antarctic ice sheets, including their ice shelves. It supports the monitoring, understanding, and prediction of ice-sheet change, delivering actionable information for research, policy, and scenario-based exploration.

[Project Website](https://dtc-ice-sheets.org/) · [Interactive Dashboard](https://dashboards.dtc-ice-sheets.org/) · [Query API](https://query.dtc-ice-sheets.org/docs)

---

The DTC Query API can provide data provenance information for every DTC-IS job.

This is a record of how the job's outputs were produced, following the [W3C PROV](https://www.w3.org/TR/prov-overview/) standard.
The provenance describes every step of the job's workflow as a dependency graph.

Depending on availability, the provenance data may include, but is not limited to, the following attributes:

### Activity Attributes

| Attribute | Vocabulary | Description |
| --- | --- | --- |
| `prov:startedAtTime` | PROV | Timestamp for the start of module execution (from the original run in case of cached results). |
| `prov:endedAtTime` | PROV | Timestamp for the end of module execution (from the original run in case of cached results). |
| `prov:hadPlan` | PROV | Identifier for the DTC-IS module template being used, e.g. data-download-template for the DDM. |
| `dtcis:command` | DTC-IS | The full command used to run the module. |
| `dtcis:dtcWrapperVersion` | DTC-IS | Current version number of the DTC Wrapper utility, which is used to transfer artifacts and metadata between Python modules. |
| `dtcis:dtcWrapperCommand` | DTC-IS | The module command wrapped by the DTC Wrapper utility. |
| `dtcis:dtcWrapperDownloadInputs` | DTC-IS | The named artifact URLs the DTC Wrapper utility was asked to make available for the command. |
| `dtcis:dtcWrapperDownloadOutputs` | DTC-IS | The named artifact paths the DTC Wrapper utility was asked to upload after command execution. |
| `dtcis:moduleName` | DTC-IS | The official DTC-IS module identifier (e.g. data_download_module). |
| `dtcis:moduleUrl` | DTC-IS | The official DTC-IS module URL, typically a GitHub Repo URL. |
| `dtcis:moduleVersion` | DTC-IS | Current version number of the DTC-IS module. |

### Entity Attributes

| Attribute | Vocabulary | Description |
| --- | --- | --- |
| `dct:identifier` | [Dublin Core Terms](https://www.dublincore.org/specifications/dublin-core/dcmi-terms/) | Unique identifier for the artifact, currently always an S3 URL. |
| `dcat:downloadURL` | [Data Catalog Vocabulary](https://www.w3.org/TR/vocab-dcat/) | The publicly accessible HTTP URL for the artifact. |

The provenance data also records which activity generated each artifact (`wasGeneratedBy`), and which artifacts
each activity used as input (`used`).


**Notebook preparations:**
To start, we need to import several libraries and pre-prepared helper functions. These are needed to interact with the DTC Query API and to illustrate the resulting data provenance information.

In [ ]:
import sys

print("⏳ Installing dependencies...")
if "google.colab" in sys.modules:
    %pip install -q xarray zarr cartopy ipywidgets==7.7.1 contextily dtc-query-client
    %pip install -q jupyter_ui_poll
    # Ensure the newest version of the helpers package is installed
    %pip uninstall -y dtc_is_notebooks
    %pip install -q --upgrade --force-reinstall --no-deps --no-cache-dir \
         git+https://github.com/DTC-Ice-Sheets/dtc_is_notebooks.git

In [ ]:
import json
import shlex
from collections import defaultdict
from datetime import datetime

import contextily as cx
import geopandas as gpd
import graphviz
import prov.model as pm
import seaborn.objects as so
import xarray as xr
from dtc_query_client import (
    CovariateAnalysisRequest,
    CovariateAnalysisType,
    GenericApi,
    IceShelfExtent,
    JobsApi,
    JobStatus,
    StateAndFateApi,
    VariableItem,
)
from dtc_query_client.helpers import iterate_job
from IPython.display import Markdown
from ipywidgets import widgets

from dtc_is_notebook_helpers.authentication_widget import request_api_token, running_in_colab
from dtc_is_notebook_helpers.cell_spinner import enable_cell_spinner, set_message
from dtc_is_notebook_helpers.data_provenance_helpers import dict_to_ipy_widget, sanitise_for_graphviz
from dtc_is_notebook_helpers.uc3_helper_functions import _load_dataset_yaml

# Show an animation at the bottom of each cell while it is running
if not running_in_colab():
    enable_cell_spinner()

## Enter DTC-IS API token

Please run the cell below, paste your DTC-IS API token into the box, and press 'Submit'. If you don’t have a token yet, you’ll need to generate one first using the link next to the box. A quick test query will check the token, and once it passes you can move on to the next cell! To submit a different token you will need to run the cell again.

---

In [ ]:
token_widget = request_api_token()
await token_widget.wait_until_authenticated()
client = token_widget.get_client()

The authenticated client object will now be passed to any DTC-Query-Client calls we make. When you reload this page, you will need to reauthenticate the client using your API key.

### Run a job via the DTC Query API

Most query API endpoints return a job ID.
Let's run a Data Linkages query as an example (see the [Data Linkages notebook for more information](./DTC-IS-DataLinkages.ipynb)):

In [ ]:
variable_mapping = _load_dataset_yaml()

variables = [
    ("ICE_VELOCITY-S1-YEARLY", "land_ice_surface_velocity_magnitude"),
    ("POLAR_PLUS_ICE_SHELVES_BASAL_MELT_TIMESERIES", "land_ice_basal_melt_rate"),
    ("GLORYS12V1", "sea_water_potential_temperature_at_sea_floor"),
]

variable_items = [
    VariableItem(
        name=(
            f"{variable_mapping[dataset]['label']}: "
            f"{variable_mapping[dataset]['variables'][variable]['label']} "
            f"[{variable_mapping[dataset]['variables'][variable]['units']}]"
        ),
        dataset=dataset,
        variable=variable,
        extent=IceShelfExtent(variable_mapping[dataset]["region"]),
    )
    for dataset, variable in variables
]

res = await StateAndFateApi(client).run_covariate_analysis(
    ice_shelf_id="thwaites",
    covariate_analysis_request=CovariateAnalysisRequest(
        start_time=datetime(1997, 1, 1),
        end_time=datetime(2021, 1, 1),
        variables=variable_items,
        analysis_type=CovariateAnalysisType.CORR,
    ),
)
job_id = res.job_id
display(Markdown(f"### Started job with ID: `{job_id}`"))

> Want to try out this notebook on a different DTC-IS job?
> Paste in your own job ID below and re-run the remaining cells:

In [ ]:
job_id_widget = widgets.Text(value=job_id, placeholder="Enter your job ID here", description="Job ID")
job_id_widget

Wait for the job to finish...

In [ ]:
async for result in iterate_job(client, job_id_widget.value):
    set_message(f"Job progress: {result.progress_done}/{result.progress_total}")

assert result.status == JobStatus.SUCCEEDED, "Job did not succeed. Status: " + str(result.status)
print("Job completed successfully!\nOutput:")
display(Markdown("```json\n" + json.dumps(result.outputs, indent=2) + "\n```"))

### Fetch data provenance information

The job's output contains many files produced during processing, but doesn't provide much information about what it all means.

Using the job ID, we can query the full data provenance information:

In [ ]:
prov_data = await JobsApi(client).job_data_provenance(job_id_widget.value)
print(json.dumps(prov_data, indent=2))

### Parse and analyse data provenance information

This data can be parsed and analysed using the [Python `prov` libary](https://prov.readthedocs.io/en/latest/).

Let's start by plotting a simplified overview of the provenance relationships between processes and outputs:

In [ ]:
doc = pm.ProvDocument.deserialize(content=json.dumps(prov_data))

dot = graphviz.Digraph("Test", graph_attr={"rankdir": "BT"})
for rec in doc.records:
    match rec:
        case pm.ProvEntity():
            url = list(rec.get_attribute("dcat:downloadURL"))[0]
            filename = url.strip().removesuffix("/").split("/")[-1]
            if "." in filename:
                filename = filename.split(".")[-1]
            else:
                filename = "folder"
            shape = {"zarr": "cylinder", "folder": "folder"}.get(filename, "note")
            color = {"zarr": "orange", "folder": "yellow"}.get(filename, "lightyellow")

            dot.node(
                sanitise_for_graphviz(rec.identifier),
                filename,
                shape=shape,
                style="filled",
                fillcolor=color,
            )
        case pm.ProvActivity():
            dot.node(
                sanitise_for_graphviz(rec.identifier),
                list(rec.get_attribute("prov:hadPlan"))[0],
                shape="box3d",
                style="filled",
                fillcolor="lightblue",
            )
        case pm.ProvUsage():
            dot.edge(sanitise_for_graphviz(rec.args[0]), sanitise_for_graphviz(rec.args[1]), "Used")
        case pm.ProvGeneration():
            dot.edge(sanitise_for_graphviz(rec.args[0]), sanitise_for_graphviz(rec.args[1]), "Was generated by")
dot

This shows how data flows through the workflow:
1. JSON files are provided as input alongside the job parameters. These are query polygons in GeoJSON format.
2. Three separate Data Download Module instances are started, each downloading data for one of the requested data variables within a query polygon. Each one produces a folder containing multiple Zarr file.
3. The Zarr files are processed into timeseries by the Timeseries entrypoint.
4. The Preprocessor entrypoint combines the timeseries into a single unified Zarr.
5. The Covariate Analyser processes the unified Zarr to produce a summary of data linkages, outputtin a JSON file as the main job output.

Each of the nodes and edges in this graph is a "Provenance Record":

- The modules are "Provenance Activities", representing a process that consumes and/or produces data.
- The artifacts are "Provenance Entities", representing a piece of data.
- The arrows coming out of entities are "Was Generated By" relationships, linking an entity to the activity that produced it.
- The arrows coming out of activities are "Was Used By" relationsships, linking an activity to its inputs.

Each of these records contains a list of attributes.
Some attributes are from standard vocabularies (`prov`, `dct`, and `dcat`), providing universally understood attributes.
Other attributes are from a new dictionary (`dtc-is`) with terms specific to DTC-IS.

The widget below allows browsing the full attributes of each record:

In [ ]:
activities = widgets.Accordion()
entities = widgets.Accordion()
relations = widgets.Accordion()

for rec in doc.records:
    match rec:
        case pm.ProvEntity():
            title = str(list(rec.get_attribute("dct:identifier"))[0])
            parent = entities
        case pm.ProvActivity():
            title = str(list(rec.get_attribute("prov:hadPlan"))[0])
            parent = activities
        case pm.ProvUsage() | pm.ProvGeneration():
            entity: pm.ProvEntity = doc.get_record(str(list(rec.get_attribute("prov:entity"))[0]))[0]
            activity: pm.ProvActivity = doc.get_record(str(list(rec.get_attribute("prov:activity"))[0]))[0]
            activity_name = list(activity.get_attribute("prov:hadPlan"))[0]
            entity_name = list(entity.get_attribute("dct:identifier"))[0]
            if isinstance(rec, pm.ProvUsage):
                title = f"'{activity_name}' used '{entity_name}'"
            else:
                title = f"'{activity_name}' generated '{entity_name}'"
            parent = relations
        case _:
            continue
    parent.children += (
        dict_to_ipy_widget({"Identifier": str(rec.identifier)} | dict(rec.attributes), special=["Identifier"]),
    )
    parent.set_title(len(parent.children) - 1, title)

tab = widgets.Tab()
tab.children = [activities, entities, relations]
tab.set_title(0, "Activities")
tab.set_title(1, "Entities")
tab.set_title(2, "Relations")
tab

In [ ]:
if job_id_widget.value != job_id:
    raise ValueError(
        "You have provided a custom job ID, congratulations! "
        "The rest of this notebook only applies to the original example job ID. "
        "Re-run the notebook from the start without a custom job ID to reset the notebook."
    )

Let's take a closer look at some of this data to see how it can be used.

Taking the `ice-shelf-state-and-fate-preprocessor` activity as an example, we can inspect the key attributes that would allow us to reproduce the exact process ourselves:

- `prov:startTime`/`prov:endTime`: Timestamps of when the calculation started and finished
- `dtcis:moduleURL`: URL to the GitHub repo containing the module code
- `dtcis:moduleVersion`: The exact version of the module used, matching a specific tag in the repo
- `dtcis:command`: The full exact command used to run the module

In [ ]:
timeseries_activity: pm.ProvActivity = doc.get_record(
    "dtcis:ice-shelf-state-and-fate-cache.6b5959ab5b3e97c95adc34c2db5bf736047008c7"
)[0]

dict_to_ipy_widget(
    {
        k: v
        for k, v in timeseries_activity.attributes
        if str(k) in ["prov:startTime", "prov:endTime", "dtcis:moduleUrl", "dtcis:moduleVersion", "dtcis:command"]
    },
)

Next, we can identify all the inputs used and outputs produced by this module, and get their download URLs:

In [ ]:
inputs = []
outputs = []
out = ""
for rec in doc.records:
    if not isinstance(rec, pm.ProvRelation):
        continue
    entity: pm.ProvEntity = doc.get_record(str(list(rec.get_attribute("prov:entity"))[0]))[0]
    activity: pm.ProvActivity = doc.get_record(str(list(rec.get_attribute("prov:activity"))[0]))[0]
    if activity.identifier != timeseries_activity.identifier:
        continue
    url = list(entity.get_attribute("dcat:downloadURL"))[0]
    if isinstance(rec, pm.ProvUsage):
        out += f"- **Used input:** {url}\n"
        inputs.append(url)
    else:
        out += f"- **Generated output:** {url}\n"
        outputs.append(url)
display(Markdown(out))

Let's load in the output URL and plot the data:

In [ ]:
ds = xr.open_dataset(outputs[0])
df = ds.to_dataframe().melt(ignore_index=False).reset_index()
metadata = ds.attrs["timeseries"]
df["variable"] = df["variable"].map(lambda x: metadata[x]["label"])
display(Markdown(f"### Aggregated with period **{ds.attrs['time_period']}**, method **{ds.attrs['aggregation']}**"))
(
    so.Plot(df, x="time", y="value")
    .share(y=False)
    .facet(row="variable")
    .add(so.Line())
    .add(so.Dot())
    .label(x="Time", y="Value")
    .layout(size=(12, 8))
    .show()
)

We can similary look at one of the Data Download Module activities for more information on where all this data comes from:

In [ ]:
ddm_activity: pm.ProvActivity = doc.get_record(
    "dtcis:ice-shelf-state-and-fate-cache.9cbdfb27b30d2b12e2b193104cef3a633381f682"
)[0]

tokens = shlex.split(list(ddm_activity.get_attribute("dtcis:command"))[0])
pairs = [(t.lstrip("-"), tokens[i + 1]) for i, t in enumerate(tokens) if t.startswith("--")]
flags = defaultdict(list)
for k, v in pairs:
    flags[k].append(v)

dataset_ids = flags["dataset_id"]
variables = flags["variables_to_download"]

dataset_overview = await GenericApi(client).dataset_overviews(dataset_ids)
variable_overview = next(v for v in dataset_overview[0].variables if v.name == variables[0])
out = f"""
### **Dataset:** {dataset_overview[0].title} (`{dataset_overview[0].dataset_id}`)
{dataset_overview[0].description}
- **Source:** `{dataset_overview[0].source}`
- **Temporal Coverage:** `{dataset_overview[0].temporal_coverage[0]}` to `{dataset_overview[0].temporal_coverage[1]}`

### **Variable:** `{variable_overview.name}`
- **Description:** {variable_overview.description}
- **Units:** ${variable_overview.units}$
- **Dimensions:** {", ".join(variable_overview.dimensions)}
- **Griddable:** {"Yes" if variable_overview.is_griddable else "No"}
"""

display(Markdown(out))

In [ ]:
coverage = gpd.read_file(dataset_overview[0].spatial_coverage.to_json())
ax = coverage.plot(alpha=0.6, color="red", figsize=(12, 6))
ax.set_title("Spatial Coverage: " + dataset_overview[0].title)
cx.add_basemap(ax, crs=coverage.crs)

Let's also plot the query polygon to see what subset of the data was included:

In [ ]:
query_json_entity: pm.ProvEntity = doc.get_record(
    "dtcis:s3-ZHRjLXF1ZXJ5LWFwaS1idWNrZXQvc3RhdGVfYW5kX2ZhdGUvaWNlX3NoZWxmX2dlb21ldHJpZXMvdGh3YWl0ZXNfb2NlYW4uanNvbg"
)[0]
query = gpd.read_file(list(query_json_entity.get_attribute("dcat:downloadURL"))[0]).to_crs(epsg=3031)
ax = query.plot(alpha=0.6, color="red", figsize=(12, 6))
ax.set_title("Data Download Query Polygon")
ax.set_xlim(-3.0e6, 0.0e6)
ax.set_ylim(-1.5e6, 1.5e6)
cx.add_basemap(ax, crs=query.crs)

## Opportunities for Future Work

This is only the tip of the iceberg for data provenance!

Future development work could add more valuable metadata and domain-specific context, such as:

- **Provenance Agents (`prov:agent`):** Attributing data and processes to specific authors or organisations.
- **Persistent Identifiers (DOIs):** Attaching Digital Object Identifiers (e.g. `dcterms:identifier`, `datacite:doi`) to published datasets and external source inputs to ensure long-term citation and traceability.
- **Geospatial Coverage (`geo:...`/`dcterms:spatial`):** Embedding coverage data directly into formalized data coverage attributes, so outputs can be indexed and discovered via automated tools in existing data catalogs.
- **Software & Code Reproducibility (`codemeta:...`/`schema:...`):** Add formalized information to aid in reproducability of results, such as exact versions of the entire tech stack (Python, R, etc.) without having to query GitHub.
- **Dataset Metadata:** Adding MIME types (`dcterms:format`), dataset sizes (`dcat:byteSize`), licences (`dcterms:license`), and other metadata useful for downstream applications.

## Conclusions

Thank you for exploring this DTC-IS demonstrator notebook. We hope it's given you a useful introduction to accessing data provenance information, and to working with the DTC-IS framework — feel free to revisit it with different variables and parameters to explore further.

We welcome your feedback, questions, and suggestions for new features. You can reach the DTC-IS team at [support@dtc-ice-sheets.org](mailto:support@dtc-ice-sheets.org).